# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanBanik/Intern_at_fly/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
from dotenv import load_dotenv
import os
import duckdb

# Load the Hugging Face token safely
load_dotenv('../../.env')
HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
SAMPLE_TABLE = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Build the feature vector
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {SAMPLE_TABLE}
    ),
    content_agg AS (
        SELECT 
            f.client_hash_id, 
            f.content_hash_id,
            SUM(CASE WHEN f.report_date BETWEEN b.end_d - INTERVAL 30 DAY AND b.end_d - INTERVAL 16 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_past15d,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_next15d,
            ANY_VALUE(c.is_deleted) as is_deleted_flag
        FROM {SAMPLE_TABLE} f
        CROSS JOIN bounds b
        LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
        GROUP BY 1, 2
        HAVING imp_past15d >= 10
    )
    SELECT 
        client_hash_id,
        content_hash_id,
        imp_past15d,
        is_deleted_flag,
        CASE WHEN imp_next15d < (imp_past15d / 3.0) * 0.85 THEN 1 ELSE 0 END AS dropped_traffic_next15d
    FROM content_agg
""").df()

print(f"Built feature vector with {len(features)} rows.")
print("Columns:", features.columns.tolist())

Built feature vector with 137455 rows.
Columns: ['client_hash_id', 'content_hash_id', 'imp_past15d', 'is_deleted_flag', 'dropped_traffic_next15d']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

*   `imp_past15d`: Impressions from Day -30 to Day -16. (Available before prediction window).
*   `is_deleted_flag`: Current deletion status from `dim_content` (Product flag - Suspect!)
*   `dropped_traffic_next15d`: Label derived strictly from Day -15 to Day 0.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Fill NaNs in the product flag assuming null means False (not deleted)
features['is_deleted_flag'] = features['is_deleted_flag'].fillna(False)

# Drop NaNs from target and features
df_test = features.dropna(subset=['imp_past15d', 'dropped_traffic_next15d']).copy()
# Convert is_deleted to numeric for the dummy model
df_test['is_deleted_numeric'] = df_test['is_deleted_flag'].astype('category').cat.codes

# We are testing if is_deleted_flag leaks the label!
X_leaky = df_test[['imp_past15d', 'is_deleted_numeric']]
X_honest = df_test[['imp_past15d']]
y = df_test['dropped_traffic_next15d']

# Proper random split for the leakage demonstration
X_l_train, X_l_test, X_h_train, X_h_test, y_train, y_test = train_test_split(
    X_leaky, X_honest, y, test_size=0.3, random_state=42
)

clf_leaky = DecisionTreeClassifier(max_depth=3).fit(X_l_train, y_train)
clf_honest = DecisionTreeClassifier(max_depth=3).fit(X_h_train, y_train)

print(f"Base rate (Majority class): {max(y_test.mean(), 1-y_test.mean()):.1%}")
print(f"Accuracy WITH product flag (Leaky): {accuracy_score(y_test, clf_leaky.predict(X_l_test)):.1%}")
print(f"Accuracy WITHOUT product flag (Honest): {accuracy_score(y_test, clf_honest.predict(X_h_test)):.1%}")

Base rate (Majority class): 88.6%
Accuracy WITH product flag (Leaky): 88.6%
Accuracy WITHOUT product flag (Honest): 88.6%


**Leakage Test Results:**
*   **Base Rate:** 88.6%
*   **Leaky Accuracy:** 88.6%
*   **Honest Accuracy:** 88.6%

*Analysis:* Our theoretical leaky feature (`is_deleted_flag`) failed to artificially inflate the score. Because this data is from the actual FlyRank database (not generated), it reflects real-world dynamics: the dataset is highly imbalanced (88.6% of pages do not drop traffic in this 15-day window) and page deletions are extremely rare. A true leaky feature ruins a model by bumping accuracy near 100%, but here the Decision Tree simply predicted the majority class every time and ignored our flag. It's a great lesson that theoretical leakage only breaks your model if it happens often enough to be exploited!

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

*   **`is_deleted` (and `last_optimized_date`)**: Excluded. These are downstream product decisions/flags. Using them allows the model to learn existing rules, not the real world.
*   **`imp_next15d`**: Excluded from features, as it overlaps with the future window we are trying to predict.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.